# 02 · 调用大模型

先把 RAG 的最后一步单独跑通：发送消息，拿到文本答案。

这一课只认识三个请求要素：模型、消息角色和 temperature。

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None
print("API 已配置" if client else "未配置 API：保留本地步骤，调用模型的单元会跳过")

未配置 API：保留本地步骤，调用模型的单元会跳过


## 最小请求

In [2]:
if client:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "用两句话解释什么是向量检索。"}],
        temperature=0,
    )
    print(response.choices[0].message.content)
else:
    print("请配置 LLM_API_KEY 或 OPENAI_API_KEY")

请配置 LLM_API_KEY 或 OPENAI_API_KEY


## system 和 user

`system` 规定模型的回答方式，`user` 提出具体问题。RAG 的“只能根据资料回答”通常放在 system 消息里。

In [3]:
messages = [
    {"role": "system", "content": "你是服饰箱包知识库助手，只根据用户提供的资料回答。"},
    {"role": "user", "content": "SKU-JK902 是什么产品？"},
]

if client:
    response = client.chat.completions.create(model=LLM_MODEL, messages=messages, temperature=0)
    print(response.choices[0].message.content)
else:
    print(messages)

[{'role': 'system', 'content': '你是服饰箱包知识库助手，只根据用户提供的资料回答。'}, {'role': 'user', 'content': 'SKU-JK902 是什么产品？'}]


## temperature

事实问答通常使用 `0`，让结果更稳定；创意任务可以调高。

In [4]:
if client:
    for temperature in [0, 1]:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": "给一个服饰品牌起名，只返回名字。"}],
            temperature=temperature,
        )
        print(temperature, response.choices[0].message.content)
else:
    print("未调用模型：temperature=0 适合 RAG 事实问答")

未调用模型：temperature=0 适合 RAG 事实问答
